# User Guide & Future Extensions

*Andrew Wang · 2026-09-08*

## Contents

1. [Introduction](#introduction)
2. [Writing Custom Strategies](#writing-custom-strategies)
3. [Censorship and Information](#censorship-and-information)
4. [Running Experiments](#running-experiments)
5. [Conclusion](#conclusion)

## Introduction

This notebook details how exactly users can implement their own strategies, run their own experiments, as well as extend/alternate the engine. 

## Writing Custom Strategies

The procedure of designing and adding a custom strategy is best show in with an example, but here are some main points to follow:

1. Custom strategies must be a subclass of the __Strategy__ parent class found in _strategies.py_, and hence must take name, ctx, and rng as arguments.
2. Must include __KEY__ attribute for identification
3. Should be written in the strategies.py file
4. action() method should return a positive integer, which is how many times to pump the balloon
Here is a simple example of a strategy that, at each balloon, rolls a fair 6-sided die to determine how many times it will pump:

NOTE: I included this strategy in this notebook as an example, users' strategies should go in strategies.py file

In [ ]:
from strategies import Strategy
from core import Balloon

class die_rolling_strat(Strategy):
    def __init__(self, name, ctx, rng):
        super().__init__(name, ctx, rng)

    def action(self, balloon: Balloon) -> int:
        threshold = self.rng.integers(low=1, high=7, size=1)
        return threshold

This is the most basic requirements of strategies. Users can also utilize the __belief_state__ dictionary that comes with the __Strategy__ parent class. This is a dictionary that contains all the colors that _can_ appear in a game and the strategy's most updated belief on how many pumps it can survive. This is a way of using memoization to make strategy decisions faster. Although all strategies I have written use a method __update_beliefs()__ to update the __belief_state__, this is not required, and the method can be named anything.

## Censorship and Information

Another important aspect of the game that the user can change is how much information that the strategies gain after each balloon. Currently, strategies have access to the payout (to both strategies), _but_ not the opponent's threshold. To change what kinds of information strategies gain from each balloon, do the following:
1. Change what the Observation object includes (this is in core.py)
2. Make apropriate changes in obs_parser() method in engine.py. This is where each player gets their designated information packets (Observation objects).

Examples of possible changes that could be interesting to examine are the following (but of course not limited to):
1. asymmetric information (one strategy gains more information than the other)
2. no payout information only if strategy popped or not
3. full information, so strategies get access to opponent's thresholds too.

## Running Experiments 
This section details how exactly to run experiments whether that is testing 10 strategies on the same set of balloons, or running a round-robin tournament with 50 strategies each playing against each other on 50 seeds of balloons.

To run an experiment, you fill the __config.yaml__ apropriately and run config.py in the __modules__ folder. The format of the .yaml file is shown:

```yaml

run_name: trial_run
master_seed: 10111

num_seeds: 10

num_balloons: 100

multiplayer: 1

strategies:
    - name: explore_exploit
      params: 
        ratio: 0.2
    - name: thompson_sampling
      params: {}

color_map:
    red: 0.2
    blue: 0.4
    orange: 0.7
```

Important rules/conventions to remember are the fllowing:
1. seeds must be integers
2. if strategies don't have any parameters, enter an empty dict as shown
3. probabilities p need to satisfy $0<p<1$
4. num_balloons > 0
5. multiplayer is 0(off) or 1(on)

If more than two strategies are entered and multiplayer mode is on, then it will automatically play a round robin tournament with the strategies. Single player experiment results will be separated into separate csv documents for each strategy (even on the same seed), and multiplayer experiments will on the same csv (the two strategies playing each other). The naming of the csv document should have sufficient information including who plays who and what seed the game is. 